# Experiment 4: Term Frequency and NER
Run the cells in order. When prompted, upload `4_1_4_2_input.txt` (used by 4.1 and 4.2).

## 4.1 Term-Frequency Analysis and NER (using toolkits: NLTK + spaCy)
- Calculate frequency of each word, save as CSV (Term, Frequency)
- Show top 10 most frequent terms
- Perform Named Entity Recognition

In [ ]:
# Install/setup toolkits
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Upload the input text file (4_1_4_2_input.txt)
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:300])

In [ ]:
# 4.1 - Term Frequency using NLTK
from nltk.tokenize import word_tokenize
from collections import Counter
import csv, string

tokens = word_tokenize(text.lower())
tokens = [t for t in tokens if t.isalpha()]   # remove punctuation/numbers

freq = Counter(tokens)

# Save output as CSV (Term, Frequency)
with open('4_1_term_frequency.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Term', 'Frequency'])
    for term, count in freq.most_common():
        writer.writerow([term, count])

# Top 10 most frequent terms
top10 = freq.most_common(10)
print('Top 10 most frequent terms:')
for term, count in top10:
    print(f'{term}: {count}')

In [ ]:
# 4.1 - Named Entity Recognition using spaCy
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(text)

print('Named Entities found:')
for ent in doc.ents:
    print(f'{ent.text:30s} -> {ent.label_}')

## 4.2 Term-Frequency Analysis (without using any toolkit)
- Same task as 4.1 (word frequency, CSV, top 10) but implemented with plain Python only, no NLTK/spaCy.

In [ ]:
# 4.2 - Term Frequency without any toolkit (pure Python)
import string
from collections import Counter
import csv

def simple_tokenize(raw_text):
    raw_text = raw_text.lower()
    for p in string.punctuation:
        raw_text = raw_text.replace(p, ' ')
    return raw_text.split()

# Reuse 'text' loaded in the 4.1 upload cell above
tokens_manual = simple_tokenize(text)
freq_manual = Counter(tokens_manual)

# Save output as CSV (Term, Frequency)
with open('4_2_term_frequency.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Term', 'Frequency'])
    for term, count in freq_manual.most_common():
        writer.writerow([term, count])

# Top 10 most frequent terms
top10_manual = freq_manual.most_common(10)
print('Top 10 most frequent terms (no toolkit):')
for term, count in top10_manual:
    print(f'{term}: {count}')

## 4.3 TF-IDF from scratch (no NLTK/spaCy/any NLP toolkit)
Computes TF, DF, IDF and TF-IDF for the three given documents and shows the top TF-IDF terms per document.

In [ ]:
# 4.3 - TF-IDF without using any NLP toolkit
import string, math
import pandas as pd

documents = [
    "Natural language processing is a field of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Machine learning is an important part of artificial intelligence."
]

def preprocess(doc):
    doc = doc.lower()
    for p in string.punctuation:
        doc = doc.replace(p, '')
    return doc.split()

tokenized_docs = [preprocess(d) for d in documents]
print('Tokenized documents:')
for i, toks in enumerate(tokenized_docs):
    print(f'Doc {i+1}: {toks}')

In [ ]:
# Step 1: Term Frequency (TF) for every term in every document
def compute_tf(tokens):
    tf = {}
    total = len(tokens)
    for t in tokens:
        tf[t] = tf.get(t, 0) + 1
    for t in tf:
        tf[t] = tf[t] / total
    return tf

tf_list = [compute_tf(tokens) for tokens in tokenized_docs]
for i, tf in enumerate(tf_list):
    print(f'\nTF - Document {i+1}:')
    for term, val in tf.items():
        print(f'  {term}: {val:.4f}')

In [ ]:
# Step 2: Document Frequency (DF) for every term
all_terms = sorted(set(term for tokens in tokenized_docs for term in tokens))
df = {}
for term in all_terms:
    df[term] = sum(1 for tokens in tokenized_docs if term in tokens)

print('DF for every term:')
for term in all_terms:
    print(f'  {term}: {df[term]}')

In [ ]:
# Step 3: Inverse Document Frequency (IDF) using logarithmic formula
N = len(documents)
idf = {term: math.log(N / df[term]) for term in all_terms}

print('IDF for every term:')
for term in all_terms:
    print(f'  {term}: {idf[term]:.4f}')

In [ ]:
# Step 4: TF-IDF for every term in every document
tfidf_list = []
for tf in tf_list:
    tfidf = {term: tf.get(term, 0) * idf[term] for term in all_terms}
    tfidf_list.append(tfidf)

# Display Term, TF, DF, IDF, TF-IDF for each document
for i, tokens in enumerate(tokenized_docs):
    print(f'\n--- Document {i+1} ---')
    rows = []
    for term in sorted(set(tokens)):
        rows.append({
            'Term': term,
            'TF': round(tf_list[i][term], 4),
            'DF': df[term],
            'IDF': round(idf[term], 4),
            'TF-IDF': round(tfidf_list[i][term], 4)
        })
    df_display = pd.DataFrame(rows).sort_values(by='TF-IDF', ascending=False)
    print(df_display.to_string(index=False))

In [ ]:
# Step 5: Top 10 terms with highest TF-IDF score, per document
for i in range(len(documents)):
    print(f'\nTop TF-IDF terms - Document {i+1}:')
    doc_terms = set(tokenized_docs[i])
    scored = [(t, tfidf_list[i][t]) for t in doc_terms]
    scored.sort(key=lambda x: x[1], reverse=True)
    for term, score in scored[:10]:
        print(f'  {term}: {score:.4f}')